# imports

In [ ]:
import torchimport torchvisionfrom torch import nnimport torchvision.transforms as transformsfrom tqdm.auto import tqdmimport warningsfrom timeit import default_timer as timerimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.metrics import confusion_matrixwarnings.filterwarnings('ignore')

## using CUDA

In [192]:
device = "cuda" if torch.cuda.is_available() else "cpu"

## Timer

In [193]:
def print_train_time(start: float, end : float, device: torch.device = None):    total_time = end - start    print(f"Train time on {device}: {total_time/60:.3f} minutes\n\n")    return total_time

## Transforming to Tensors

In [194]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

In [195]:
trainset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)testset = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

## 'Train', 'Validation', 'Test'

In [196]:
train_size = int(0.8 * len(trainset))val_size = len(trainset) - train_sizetrain_subset, val_subset = torch.utils.data.random_split(trainset, [train_size, val_size])# Create DataLoaders for each settrainloader = torch.utils.data.DataLoader(train_subset, batch_size=64, shuffle=True)valloader = torch.utils.data.DataLoader(val_subset, batch_size=64, shuffle=False)testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

## Deep Neural Network

In [197]:
class DeepNN(nn.Module):    def __init__(self, input_features=784, output_features=10, hidden_features=104, init_method=None):        super().__init__()        self.layer_stack = nn.Sequential(            nn.Linear(in_features=input_features, out_features=hidden_features),            nn.ReLU(),            nn.Linear(in_features=hidden_features, out_features=hidden_features),            nn.ReLU(),            nn.Linear(in_features=hidden_features, out_features=hidden_features),            nn.ReLU(),            nn.Linear(in_features=hidden_features, out_features=hidden_features),            nn.ReLU(),            nn.Linear(in_features=hidden_features, out_features=output_features),        )        self._initialize_weights(init_method)    def _initialize_weights(self, init_method):        if init_method != None:            for layer in self.layer_stack:                if isinstance(layer, nn.Linear):                    if init_method == 'he':                        nn.init.kaiming_normal_(layer.weight, mode='fan_in', nonlinearity='relu')                    elif init_method == 'xavier':                        nn.init.xavier_normal_(layer.weight)                    elif init_method == 'random':                        nn.init.normal_(layer.weight, mean=0, std=1)                    nn.init.zeros_(layer.bias)  # Initialize biases to zero    def forward(self, x):        return self.layer_stack(x)

## Wide Nerual Network

In [198]:
class WideNN(nn.Module):    def __init__(self, input_features=784, output_features=10, hidden_features=136, init_method=None):        super().__init__()        self.layer_stack = nn.Sequential(            nn.Linear(in_features=input_features, out_features=hidden_features),            nn.ReLU(),            nn.Linear(in_features=hidden_features, out_features=output_features),        )        self._initialize_weights(init_method)    def _initialize_weights(self, init_method):        if init_method != None:            for layer in self.layer_stack:                if isinstance(layer, nn.Linear):                    if init_method == 'he':                        nn.init.kaiming_normal_(layer.weight, mode='fan_in', nonlinearity='relu')                    elif init_method == 'xavier':                        nn.init.xavier_normal_(layer.weight)                    elif init_method == 'random':                        nn.init.normal_(layer.weight, mean=0, std=1)                    nn.init.zeros_(layer.bias)  # Initialize biases to zero    def forward(self, x):        return self.layer_stack(x)

In [199]:
deep_model = DeepNN().to(device)wide_model = WideNN().to(device)

In [ ]:
print(f"Deep_model state_dict {deep_model.state_dict}")print(f"Wide_model state_dict {wide_model.state_dict}")

Deep_model state_dict <bound method Module.state_dict of DeepNN(
  (layer_stack): Sequential(
    (0): Linear(in_features=784, out_features=104, bias=True)
    (1): ReLU()
    (2): Linear(in_features=104, out_features=104, bias=True)
    (3): ReLU()
    (4): Linear(in_features=104, out_features=104, bias=True)
    (5): ReLU()
    (6): Linear(in_features=104, out_features=104, bias=True)
    (7): ReLU()
    (8): Linear(in_features=104, out_features=10, bias=True)
  )
)>
Deep_model state_dict <bound method Module.state_dict of WideNN(
  (layer_stack): Sequential(
    (0): Linear(in_features=784, out_features=136, bias=True)
    (1): ReLU()
    (2): Linear(in_features=136, out_features=10, bias=True)
  )
)>


## 'Loss' and 'Optimizer'

In [201]:
deep_loss = nn.CrossEntropyLoss()wide_loss = nn.CrossEntropyLoss()deep_optimizer = torch.optim.Adam(deep_model.parameters(), lr=0.001)wide_optimizer = torch.optim.Adam(wide_model.parameters(), lr=0.001)

## "Train" and "Evaluation" Loop

In [ ]:
def train_and_evaluate(model, trainloader, valloader, loss_fn, optimizer, device, epochs=8):    start_time = timer()    train_losses = []    eval_losses = []    for epoch in range(epochs):        model.train()        train_loss = 0        for images, labels in tqdm(trainloader, desc=f"Epoch {epoch+1}/{epochs} - Training"):            images, labels = images.to(device), labels.to(device)            images = images.flatten(start_dim=1)            outputs = model(images)            loss = loss_fn(outputs, labels)            optimizer.zero_grad()            loss.backward()            optimizer.step()            train_loss += loss.item()        avg_train_loss = train_loss/len(trainloader)        train_losses.append(avg_train_loss)        print(f" ** Training Loss: {avg_train_loss:.4f}")        model.eval()        correct = 0        total = 0        eval_loss = 0        with torch.inference_mode():            for images, labels in tqdm(valloader, desc=f"Epoch {epoch+1}/{epochs} - Evaluation"):                images, labels = images.to(device), labels.to(device)                images = images.flatten(start_dim=1)                outputs = model(images)                predicted = torch.argmax(outputs, dim=1)                total += labels.size(0)                correct += (predicted == labels).sum().item()                loss = loss_fn(outputs, labels)  # Calculate evaluation loss                eval_loss += loss.item()        avg_eval_loss = eval_loss / len(valloader)        eval_losses.append(avg_eval_loss)        accuracy = 100 * correct / total        print(f" ** Evaluation Accuracy: {accuracy:.2f}%\n\n")    end_time = timer()    print_train_time(start_time, end_time, device=device)    return train_losses, eval_losses

## Evaluation on Test set

In [203]:
def testset_prediction(model, testloader, device):    model.eval()    all_preds = []    all_labels = []    correct = 0    total = 0    with torch.inference_mode():        for images, labels in tqdm(testloader, desc="Final Test Evaluation"):            images, labels = images.to(device), labels.to(device)            images = images.flatten(start_dim=1)            outputs = model(images)            predicted = torch.argmax(outputs, dim=1)            total += labels.size(0)            correct += (predicted == labels).sum().item()            all_preds.extend(predicted.cpu().numpy())            all_labels.extend(labels.cpu().numpy())    test_accuracy = 100 * correct / total    conf_matrix = confusion_matrix(all_labels, all_preds)    print(f"Final Test Accuracy: {test_accuracy:.2f}%")    return conf_matrix, test_accuracy

## visualizer

In [204]:
def plot_train_eval(train_losses, eval_losses, confusion_matrix):    fig, axes = plt.subplots(1, 2, figsize=(20, 10))    plt.sca(axes[0])    plt.plot(train_losses, label="Training Loss", marker='o')    plt.plot(eval_losses, label="Evaluation Loss", marker='s')    plt.xlabel("Epoch")    plt.ylabel("Loss")    plt.title("Training and Evaluation Loss")    plt.legend()    plt.sca(axes[1])    sns.heatmap(confusion_matrix.astype(int), vmin = 0, vmax = 1,center = 0, cmap='crest',  square = True, annot = True, cbar=False, linewidth=.5, fmt='d',                annot_kws={"size": 18,})    plt.title("Confusion Matrix", fontsize=8)    plt.tight_layout()    plt.show()